In [16]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import polars as pl
from pathlib import Path
import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


In [17]:
customers_dataset = pd.read_csv('data/olist_customers_dataset.csv')
display(customers_dataset.head())

def clean_data(customers_dataset):
    # Capitalize the first character in column: 'customer_city'
    customers_dataset['customer_city'] = customers_dataset['customer_city'].str.title()
    return customers_dataset

customers_dataset_clean = clean_data(customers_dataset.copy())
customers_dataset_clean.head()



FileNotFoundError: [Errno 2] No such file or directory: 'data/olist_customers_dataset.csv'

In [ ]:

geolocation_dataset = pd.read_parquet('data/olist_geolocation_dataset.parquet', engine='pyarrow')
display(geolocation_dataset.head())



def clean_data(geolocation_dataset):
    # Capitalize the first character in column: 'geolocation_city'
    geolocation_dataset['geolocation_city'] = geolocation_dataset['geolocation_city'].str.title()
    return geolocation_dataset

geolocation_dataset_clean = clean_data(geolocation_dataset.copy())
geolocation_dataset_clean.head()


In [ ]:
order_items_dataset = pd.read_csv('data/olist_order_items_dataset.csv')
display(order_items_dataset.head())

def clean_data(order_items_dataset):
    # Change column type to datetime64[ns] for column: 'shipping_limit_date'
    order_items_dataset = order_items_dataset.astype({'shipping_limit_date': 'datetime64[ns]'})
    return order_items_dataset

order_items_dataset_clean = clean_data(order_items_dataset.copy())
order_items_dataset_clean.head()



In [ ]:
order_payments_dataset = pd.read_csv('data/olist_order_payments_dataset.csv')
display(order_payments_dataset.head())

def clean_data(order_payments_dataset):
    # Replace underscores with spaces in payment_type
    order_payments_dataset['payment_type'] = order_payments_dataset['payment_type'].str.replace('_', ' ')
    # Capitalize the first character in column: 'payment_type'
    order_payments_dataset['payment_type'] = order_payments_dataset['payment_type'].str.title()
    return order_payments_dataset

order_payments_dataset_clean = clean_data(order_payments_dataset.copy())
order_payments_dataset_clean.head()




In [ ]:
order_reviews_dataset = pd.read_csv('data/olist_order_reviews_dataset.csv')
display(order_reviews_dataset.head())

def clean_data(order_reviews_dataset):
    # Capitalize the first character in column: 'review_comment_title'
    order_reviews_dataset['review_comment_title'] = order_reviews_dataset['review_comment_title'].str.capitalize()
    # Capitalize the first character in column: 'review_comment_message'
    order_reviews_dataset['review_comment_message'] = order_reviews_dataset['review_comment_message'].str.capitalize()
    # Assuming order_reviews_dataset is already defined
    order_reviews_dataset['review_creation_date'] = pd.to_datetime(order_reviews_dataset['review_creation_date'])
    order_reviews_dataset['review_answer_timestamp'] = pd.to_datetime(order_reviews_dataset['review_answer_timestamp'])
    return order_reviews_dataset

order_reviews_dataset_clean = clean_data(order_reviews_dataset.copy())
order_reviews_dataset_clean.head()



In [ ]:
orders_dataset = pd.read_csv('data/olist_orders_dataset.csv')
display(orders_dataset.head())

def clean_data(orders_dataset):
    # Change column type to datetime64[ns] for column: 'order_purchase_timestamp'
    orders_dataset = orders_dataset.astype({'order_purchase_timestamp': 'datetime64[ns]'})
    # Assuming orders_dataset is already defined
    orders_dataset['order_approved_at'] = pd.to_datetime(orders_dataset['order_approved_at'])
    orders_dataset['order_delivered_carrier_date'] = pd.to_datetime(orders_dataset['order_delivered_carrier_date'])
    orders_dataset['order_delivered_customer_date'] = pd.to_datetime(orders_dataset['order_delivered_customer_date'])
    orders_dataset['order_estimated_delivery_date'] = pd.to_datetime(orders_dataset['order_estimated_delivery_date'])
    # Capitalize the first character in column: 'order_status'
    orders_dataset['order_status'] = orders_dataset['order_status'].str.capitalize()
    return orders_dataset

orders_dataset_clean = clean_data(orders_dataset.copy())
orders_dataset_clean.head()


In [ ]:
products_dataset = pd.read_csv('data/olist_products_dataset.csv')
display(products_dataset.head())

def clean_data(products_dataset):
    # Replace "_" with space and capitalize
    products_dataset['product_category_name'] = products_dataset['product_category_name'].str.replace('_', ' ').str.title()
    return products_dataset

products_dataset_clean = clean_data(products_dataset.copy())
products_dataset_clean.head()


In [ ]:

sellers_dataset = pd.read_csv('data/olist_sellers_dataset.csv')
display(sellers_dataset.head())

def clean_data(sellers_dataset):
    # Capitalize the first character in column: 'seller_city'
    sellers_dataset['seller_city'] = sellers_dataset['seller_city'].str.title()
    return sellers_dataset

sellers_dataset_clean = clean_data(sellers_dataset.copy())
sellers_dataset_clean.head()


In [ ]:
product_category_name_translation = pd.read_csv('data/product_category_name_translation.csv')
display(product_category_name_translation.head())

def clean_data(product_category_name_translation):
    # Assuming product_category_name_translation is a DataFrame
    product_category_name_translation['product_category_name'] = product_category_name_translation['product_category_name'].str.replace('_', ' ').str.title()
    product_category_name_translation['product_category_name_english'] = product_category_name_translation['product_category_name_english'].str.replace('_', ' ').str.title()
    return product_category_name_translation

product_category_name_translation_clean = clean_data(product_category_name_translation.copy())
product_category_name_translation_clean.head()



# 01 - Olist dataset discovery (EDA)

**Purpose.** Establish what the nine raw Olist files contain *before* modelling them: keys and grain, referential integrity, join fan-out, nulls and anomalies, and the distributions that matter. Every section ends with a finding, and the last section turns those findings into decisions for the star schema (`sql/`, `docs/data/DATA_MODEL.md`).

**Conventions**
- Raw files are only *read* (through DuckDB views); nothing is cleaned, re-cased or dropped here. Anything odd is documented, not silently removed.
- Zip-code prefixes are forced to `VARCHAR` so leading zeros survive.
- Wording on relationships is associational ("late delivery is associated with lower review scores"), not causal.


In [ ]:


pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)

# Works whether the notebook is run from the repo root or from notebooks/
DATA = Path.cwd().parent / "data" if Path.cwd().name == "notebooks" else Path.cwd() / "data"

SOURCES = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "translation": "product_category_name_translation.csv",
}
ZIP_TYPES = {
    "customers": "{'customer_zip_code_prefix': 'VARCHAR'}",
    "sellers": "{'seller_zip_code_prefix': 'VARCHAR'}",
}

con = duckdb.connect()
for name, file in SOURCES.items():
    types = f", types={ZIP_TYPES[name]}" if name in ZIP_TYPES else ""
    con.execute(
        f"create or replace view {name} as "
        f"select * from read_csv_auto('{(DATA / file).as_posix()}', header=true{types})"
    )
con.execute(
    "create or replace view geo as "
    f"select * from read_parquet('{(DATA / 'olist_geolocation_dataset.parquet').as_posix()}')"
)
TABLES = [*SOURCES, "geo"]


def q(sql: str) -> pd.DataFrame:
    return con.sql(sql).df()


# Chart styling: validated blue/orange pair, recessive grid, thin marks, no top/right spines
SURFACE, TEXT, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
BLUE, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": GRID,
    "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "text.color": TEXT,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "figure.dpi": 110,
    "axes.titlelocation": "left", "axes.titlesize": 11, "axes.titleweight": "bold",
    "legend.frameon": False,
})
print("data dir:", DATA)

## 1. Inventory
Row counts, column types and null percentages for all nine files.

In [ ]:
counts = pd.DataFrame(
    [(t, con.sql(f"select count(*) from {t}").fetchone()[0], len(q(f"describe {t}"))) for t in TABLES],
    columns=["table", "rows", "columns"],
)
display(counts)

prof = pd.concat([q(f"summarize {t}").assign(table=t) for t in TABLES], ignore_index=True)
prof["null_percentage"] = prof["null_percentage"].astype(float)
prof = prof[["table", "column_name", "column_type", "null_percentage", "approx_unique"]]
display(prof)

In [ ]:
print("Columns with nulls:")
display(prof[prof.null_percentage > 0].sort_values("null_percentage", ascending=False))
print("translation columns:", q("describe translation").column_name.tolist(), "(BOM handled if the first name is clean)")

## 2. Keys and grain
Which column combinations are really unique? The grain of each table decides the grain of its fact.

In [ ]:
KEYS = [
    ("orders", "order_id"), ("customers", "customer_id"),
    ("items", "order_id, order_item_id"), ("payments", "order_id, payment_sequential"),
    ("reviews", "review_id"), ("reviews", "review_id, order_id"),
    ("products", "product_id"), ("sellers", "seller_id"),
    ("translation", "product_category_name"), ("geo", "geolocation_zip_code_prefix"),
]
out = []
for t, k in KEYS:
    n, d = con.sql(f"select count(*), count(distinct ({k})) from {t}").fetchone()
    out.append((t, k, n, d, n == d))
display(pd.DataFrame(out, columns=["table", "candidate_key", "rows", "distinct_keys", "is_unique"]))

In [ ]:
print("customer_id is per-order; customer_unique_id is the real customer:")
display(q("select count(*) customer_ids, count(distinct customer_unique_id) unique_customers from customers"))
print("Orders per unique customer:")
display(q("""
    select orders_per_customer, count(*) customers
    from (select c.customer_unique_id, count(*) orders_per_customer
          from orders o join customers c using (customer_id) group by 1)
    group by 1 order by 1"""))
print("Unique customers whose zip prefix or state changes between orders:")
display(q("""
    select count(*) customers_with_changing_location from (
        select customer_unique_id from customers group by 1
        having count(distinct customer_zip_code_prefix) > 1 or count(distinct customer_state) > 1)"""))

print("Reviews:")
display(q("""
    select count(*) review_rows, count(distinct review_id) distinct_review_ids,
           count(distinct order_id) orders_with_review,
           (select count(*) from (select review_id from reviews group by 1
                                  having count(distinct order_id) > 1)) review_ids_on_multiple_orders
    from reviews"""))
display(q("""
    select reviews_per_order, count(*) orders
    from (select order_id, count(*) reviews_per_order from reviews group by 1)
    group by 1 order by 1"""))
print("Items / payments per order:")
display(q("""
    select (select max(order_item_id) from items) max_items_per_order,
           (select count(*) from items where order_item_id > 1) rows_in_multi_item_orders,
           (select max(c) from (select count(*) c from payments group by order_id)) max_payments_per_order,
           (select count(*) from (select order_id from payments group by 1 having count(*) > 1)) orders_with_multiple_payments,
           (select count(*) from (select order_id from items group by 1 having count(distinct seller_id) > 1)) multi_seller_orders"""))

## 3. Referential integrity
Orphans in both directions: children pointing at a missing parent, and parents with no children.

In [ ]:
CHILD_TO_PARENT = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("items", "order_id", "orders", "order_id"),
    ("items", "product_id", "products", "product_id"),
    ("items", "seller_id", "sellers", "seller_id"),
    ("payments", "order_id", "orders", "order_id"),
    ("reviews", "order_id", "orders", "order_id"),
]
PARENT_WITHOUT_CHILD = [
    ("customers", "customer_id", "orders"), ("orders", "order_id", "items"),
    ("orders", "order_id", "payments"), ("orders", "order_id", "reviews"),
    ("products", "product_id", "items"), ("sellers", "seller_id", "items"),
]
rows = []
for c, ck, p, pk in CHILD_TO_PARENT:
    n = con.sql(f"select count(*) from {c} where {ck} not in (select {pk} from {p})").fetchone()[0]
    rows.append((f"{c}.{ck}", f"{p}.{pk}", "orphan child rows", n))
for p, pk, c in PARENT_WITHOUT_CHILD:
    n = con.sql(f"select count(*) from {p} where {pk} not in (select {pk} from {c})").fetchone()[0]
    rows.append((f"{p}.{pk}", f"{c}.{pk}", "parents with no child", n))
display(pd.DataFrame(rows, columns=["column", "references", "check", "count"]))

print("Orders without any item, by status:")
display(q("""select order_status, count(*) orders from orders
             where order_id not in (select order_id from items) group by 1 order by 2 desc"""))

## 4. Fan-out demonstration
Items, payments and reviews all hang off `order_id` at *different* grains. Joining them side by side repeats each item once per payment and once per review, so any item-level measure is over-counted. Left joins are used so only fan-out (not row loss) is measured.

In [ ]:
true_price = con.sql("select sum(price) from items").fetchone()[0]
base = "from orders o left join items i using (order_id)"
variants = {
    "orders": "select count(*), null::double from orders",
    "+ items": f"select count(*), sum(i.price) {base}",
    "+ items + payments": f"select count(*), sum(i.price) {base} left join payments p using (order_id)",
    "+ items + payments + reviews": (
        f"select count(*), sum(i.price) {base} left join payments p using (order_id) "
        "left join reviews r using (order_id)"
    ),
}
rows = []
for label, sql in variants.items():
    n, s = con.sql(sql).fetchone()
    rows.append((label, n, s, None if s is None else s / true_price - 1))
fan = pd.DataFrame(rows, columns=["join", "rows", "sum_item_price", "overstatement_vs_true"])
display(fan.style.format({"rows": "{:,.0f}", "sum_item_price": "{:,.2f}", "overstatement_vs_true": "{:.2%}"}, na_rep="-"))
print(f"True sum of item price: {true_price:,.2f}")

In [ ]:
# Correct pattern: bring every child to the order grain first, then join one-to-one.
safe = q("""
    with i as (select order_id, count(*) n_items, sum(price) price, sum(freight_value) freight from items group by 1),
         p as (select order_id, count(*) n_payments, sum(payment_value) paid from payments group by 1),
         r as (select order_id, count(*) n_reviews, avg(review_score) avg_score from reviews group by 1)
    select count(*) n_rows, sum(price) sum_item_price from orders o
    left join i using (order_id) left join p using (order_id) left join r using (order_id)""")
display(safe)
assert safe.n_rows[0] == con.sql("select count(*) from orders").fetchone()[0]
assert abs(safe.sum_item_price[0] - true_price) < 0.01
print("Pre-aggregating each child to the order grain keeps one row per order and the exact item total.")

## 5. Nulls and anomalies

In [ ]:
print("Missing timestamps by order status:")
display(q("""
    select order_status, count(*) orders,
           sum((order_approved_at is null)::int) no_approved_at,
           sum((order_delivered_carrier_date is null)::int) no_carrier_date,
           sum((order_delivered_customer_date is null)::int) no_customer_date
    from orders group by 1 order by 2 desc"""))
print("Timestamp ordering and lateness:")
display(q("""
    select sum((order_approved_at < order_purchase_timestamp)::int) approved_before_purchase,
           sum((order_delivered_carrier_date < order_purchase_timestamp)::int) carrier_before_purchase,
           sum((order_delivered_customer_date < order_delivered_carrier_date)::int) customer_before_carrier,
           sum((order_delivered_customer_date < order_purchase_timestamp)::int) customer_before_purchase,
           sum((order_delivered_customer_date > order_estimated_delivery_date)::int) delivered_late,
           sum((order_delivered_customer_date is not null)::int) with_delivery_date
    from orders"""))
print("Non-delivered orders that nevertheless have a customer delivery date:")
display(q("""select order_status, count(*) orders from orders
             where order_delivered_customer_date is not null and order_status <> 'delivered' group by 1"""))

In [ ]:
print("Products:")
display(q("""
    select count(*) products, sum((product_category_name is null)::int) null_category,
           sum((product_photos_qty is null)::int) null_photos_and_text_lengths,
           sum((product_weight_g is null)::int) null_weight, sum((product_weight_g = 0)::int) zero_weight,
           sum((product_length_cm is null)::int) null_dimensions from products"""))
print("Categories present in products but missing from the translation file:")
display(q("""select product_category_name, count(*) products from products
             where product_category_name is not null
               and product_category_name not in (select product_category_name from translation)
             group by 1"""))
print("Item money columns:")
display(q("""select min(price) min_price, median(price) median_price, max(price) max_price,
                    sum((freight_value = 0)::int) zero_freight_rows, max(freight_value) max_freight from items"""))
print("Payments by type:")
display(q("""select payment_type, count(*) payments, min(payment_value) min_value, max(payment_value) max_value,
                    sum((payment_value = 0)::int) zero_value, max(payment_installments) max_installments,
                    sum((payment_installments = 0)::int) zero_installments
             from payments group by 1 order by 2 desc"""))

## 6. Reconciliation: item totals vs payments
Does `sum(price + freight)` per order equal `sum(payment_value)`? This decides whether payments can be treated as a second view of the same money.

In [ ]:
rec = q("""
    with i as (select order_id, sum(price + freight_value) items_total from items group by 1),
         p as (select order_id, sum(payment_value) paid, count(*) n_payments,
                      bool_or(payment_type = 'voucher') has_voucher, max(payment_installments) max_installments
               from payments group by 1)
    select order_id, items_total, paid, paid - items_total diff, n_payments, has_voucher, max_installments
    from i join p using (order_id)""")
rec["matches"] = rec["diff"].abs() <= 0.01
print(f"orders compared: {len(rec):,}; match within 1 cent: {rec.matches.sum():,}; mismatch: {(~rec.matches).sum():,}")
print(f"sum items_total {rec.items_total.sum():,.0f} vs sum paid {rec.paid.sum():,.0f}")
display(rec.groupby("has_voucher").matches.agg(orders="size", match_rate="mean"))
display(rec.groupby(rec.max_installments > 1).matches.agg(orders="size", match_rate="mean").rename_axis("installments > 1"))
display(rec.loc[~rec.matches, "diff"].describe())
display(rec.loc[~rec.matches].assign(abs_diff=lambda d: d["diff"].abs()).sort_values("abs_diff", ascending=False).head(5))

## 7. Distributions and time

In [ ]:
print("Order status mix:")
display(q("select order_status, count(*) orders, count(*) * 1.0 / sum(count(*)) over () share from orders group by 1 order by 2 desc"))

monthly = q("""select date_trunc('month', order_purchase_timestamp) month, count(*) orders
               from orders group by 1 order by 1""")
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(monthly.month, monthly.orders, color=BLUE, lw=2, marker="o", ms=4)
ax.set_title("Orders per month: ramp-up from 2017, near-empty edges (2016-09 to 2016-12, 2018-09 to 2018-10)")
ax.set_ylabel("orders"); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()
print("Purchase range:", q("select min(order_purchase_timestamp) first_order, max(order_purchase_timestamp) last_order from orders").iloc[0].tolist())

In [ ]:
deliv = q("""
    select date_diff('day', order_purchase_timestamp, order_delivered_customer_date) days_to_deliver,
           date_diff('day', order_delivered_customer_date, order_estimated_delivery_date) days_early
    from orders where order_delivered_customer_date is not null""")
print(deliv.days_to_deliver.describe().round(1).to_string())
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.hist(deliv.days_to_deliver.clip(upper=60), bins=range(0, 62), color=BLUE, edgecolor=SURFACE, linewidth=0.6)
med = deliv.days_to_deliver.median()
ax.axvline(med, color=MUTED, lw=1, ls="--")
ax.text(med + 0.8, ax.get_ylim()[1] * 0.92, f"median {med:.0f} days", color=MUTED)
ax.set_title("Days from purchase to delivery (values above 60 grouped in the last bar)")
ax.set_xlabel("days"); ax.set_ylabel("orders"); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

In [ ]:
rv = q("""
    select r.review_score,
           case when o.order_delivered_customer_date <= o.order_estimated_delivery_date then 'on time' else 'late' end outcome,
           count(*) reviews
    from reviews r join orders o using (order_id)
    where o.order_delivered_customer_date is not null group by all""")
share = rv.pivot(index="review_score", columns="outcome", values="reviews").fillna(0)
share = share / share.sum()
avg = rv.assign(w=rv.review_score * rv.reviews).groupby("outcome").agg(w=("w", "sum"), n=("reviews", "sum"))
print("Average review score:", (avg.w / avg.n).round(2).to_dict(), "| reviews:", avg.n.to_dict())

fig, ax = plt.subplots(figsize=(9, 3.4))
x = share.index.to_numpy(); w = 0.38
ax.bar(x - w / 2, share["on time"], width=w - 0.03, color=BLUE, label="delivered on time")
ax.bar(x + w / 2, share["late"], width=w - 0.03, color=ORANGE, label="delivered late")
ax.set_title("Review score mix: late deliveries are associated with lower scores")
ax.set_xlabel("review score"); ax.set_ylabel("share of reviews"); ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.grid(axis="x", visible=False); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
cat = q("""
    select coalesce(t.product_category_name_english, p.product_category_name, 'UNKNOWN') category,
           count(*) items, sum(i.price) revenue
    from items i join products p using (product_id)
    left join translation t using (product_category_name)
    group by 1 order by 3 desc""")
top = cat.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.barh(top.category, top.revenue, color=BLUE, height=0.65)
for y, v in enumerate(top.revenue):
    ax.text(v, y, f" {v / 1e3:,.0f}k", va="center", color=MUTED, fontsize=9)
ax.set_title(f"Top 10 categories by item revenue ({len(cat)} categories incl. UNKNOWN)")
ax.set_xlabel("sum of item price (BRL)"); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()
print("Share of revenue in top 10:", f"{cat.head(10).revenue.sum() / cat.revenue.sum():.1%}")
print("Customers by state (top 8):")
display(q("select customer_state, count(*) customers from customers group by 1 order by 2 desc limit 8"))

## 8. Geolocation
The geolocation file has ~1M rows for ~19k zip prefixes. It needs collapsing to one row per prefix before it can be a dimension.

In [ ]:
display(q("""
    select count(*) points, count(distinct geolocation_zip_code_prefix) prefixes,
           sum((geolocation_lat not between -34 and 6 or geolocation_lng not between -74 and -28)::int) outside_brazil_bbox,
           (select count(*) from (select geolocation_zip_code_prefix from geo group by 1
                                  having count(distinct geolocation_state) > 1)) prefixes_in_multiple_states
    from geo"""))
con.execute("""
    create or replace view geo_by_zip as
    select geolocation_zip_code_prefix zip_prefix,
           median(geolocation_lat) lat, median(geolocation_lng) lng,
           mode(geolocation_state) state, mode(geolocation_city) city, count(*) n_points
    from geo
    where geolocation_lat between -34 and 6 and geolocation_lng between -74 and -28
    group by 1""")
display(q("select count(*) prefixes, min(n_points) min_points, median(n_points) median_points, max(n_points) max_points from geo_by_zip"))
display(q("""
    select 'customers' entity, count(*) rows, sum((customer_zip_code_prefix in (select zip_prefix from geo_by_zip))::int) rows_matched,
           count(distinct customer_zip_code_prefix) distinct_prefixes,
           count(distinct customer_zip_code_prefix) filter (where customer_zip_code_prefix not in (select zip_prefix from geo_by_zip)) prefixes_unmatched
    from customers
    union all
    select 'sellers', count(*), sum((seller_zip_code_prefix in (select zip_prefix from geo_by_zip))::int),
           count(distinct seller_zip_code_prefix),
           count(distinct seller_zip_code_prefix) filter (where seller_zip_code_prefix not in (select zip_prefix from geo_by_zip))
    from sellers"""))